# TranSTR + DINOv3 + GroundingDINO-SigLIP2 + Top-5 Temporal Relation + NCOD LUM+HUM — Colab Pro
**Object feature mới**: SigLIP2 ROI 768D + DeBERTa class 768D + bbox 4D = 1540D. Data tải từ Kaggle; bs=32, không gradient accumulation, 20 epochs, EMA weights cho eval/save.


In [ ]:
# CELL 1: Git Clone & Setup
import os
REPO_URL  = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH    = 'full_mode'

if not os.path.exists(REPO_NAME):
    print(f'Cloning {REPO_URL}...')
    os.system(f'git clone {REPO_URL} -b {BRANCH}')
else:
    print('Repo already exists.')

target_dir = os.path.join(os.getcwd(), REPO_NAME, 'causalvid')
if os.path.exists(target_dir) and os.path.basename(os.getcwd()) != 'causalvid':
    os.chdir(target_dir)
    print(f'Changed directory to: {os.getcwd()}')
else:
    print(f'Working directory: {os.getcwd()}')

In [ ]:
%cd /content/tranSTR_Casual

In [ ]:
# CELL 2: Install dependencies & W&B login (W&B-only)
print('=== CELL 2: Install & W&B Setup ===')
os.system('pip install -q wandb kaggle kaggle-hub --upgrade')

import os
import wandb

WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ENTITY = None
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')

try:
    from google.colab import userdata
    WANDB_API_KEY = userdata.get('WANDB_API_KEY') or WANDB_API_KEY
except Exception:
    pass

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY, relogin=True)
else:
    # Uses an existing netrc/session; otherwise W&B will prompt once.
    wandb.login()
print('✅ W&B logged in — checkpoints and results use W&B Artifacts only.')


In [ ]:
# CELL 3: Setup creds + download data
#  - DINOv3 frame features → Kaggle (giữ nguyên)
#  - GroundingDINO+SigLIP2 object features → Kaggle → resolve/merge
#  - Annotations + splits → Kaggle
print('=== CELL 3: Data Download ===')

import os
import json
import glob
import shutil
import zipfile
from pathlib import Path
from collections import Counter as _Counter
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

!pip install -q -U kagglehub kaggle wandb

# ============================================
# Kaggle creds (Colab Secrets → fallback kaggle.json)
# ============================================
creds_ok = False
try:
    from google.colab import userdata
    username = userdata.get('KAGGLE_USERNAME')
    key = userdata.get('KAGGLE_KEY')
    if username and key:
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = key
        creds_ok = True
        print('Kaggle credentials loaded from Colab Secrets')
except Exception as e:
    print(f'Could not load Kaggle from Colab Secrets: {e}')

if not creds_ok:
    kaggle_json_candidates = [
        '/content/kaggle.json',
        '/content/.kaggle/kaggle.json',
        os.path.expanduser('~/.kaggle/kaggle.json'),
    ]
    found = next((p for p in kaggle_json_candidates if os.path.exists(p)), None)
    if found is None:
        raise FileNotFoundError('Kaggle creds not found.')
    with open(found, 'r', encoding='utf-8') as f:
        kg = json.load(f)
    os.environ['KAGGLE_USERNAME'] = kg.get('username', '')
    os.environ['KAGGLE_KEY'] = kg.get('key', '')
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp {found} /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    print(f'Kaggle credentials loaded from: {found}')

import kagglehub

def _find_best_dir(root_path, preferred_dir_names):
    root = Path(root_path)
    if not root.exists():
        return None
    for name in preferred_dir_names:
        exact = [p for p in root.rglob('*') if p.is_dir() and p.name == name]
        if exact:
            return str(exact[0])
    npy_counts = {}
    for p in root.rglob('*.npy'):
        npy_counts[str(p.parent)] = npy_counts.get(str(p.parent), 0) + 1
    if npy_counts:
        return max(npy_counts.items(), key=lambda x: x[1])[0]
    pt_counts = {}
    for p in root.rglob('*.pt'):
        pt_counts[str(p.parent)] = pt_counts.get(str(p.parent), 0) + 1
    if pt_counts:
        return max(pt_counts.items(), key=lambda x: x[1])[0]
    return str(root)

print('\n--- Kaggle datasets (DINOv3 + annotations + splits) ---')
DINOV3_HUB_PATH = kagglehub.dataset_download('danielq07/dinov3-feat')
print(f'DINOv3 features root: {DINOV3_HUB_PATH}')

ANN_HUB_PATH = kagglehub.dataset_download('lusnaw/text-annotation')
print(f'Annotations root: {ANN_HUB_PATH}')

SPLIT_HUB_PATH = kagglehub.dataset_download('danielq07/casual-vid-data-split')
print(f'Splits root: {SPLIT_HUB_PATH}')

# ============================================
# GroundingDINO + SigLIP2 features from Kaggle
# https://www.kaggle.com/datasets/danielq07/CausalVQA-Groundingdino-SigLIP-feature
# Object schema: SigLIP2 ROI 768 + DeBERTa class 768 + bbox 4 = 1540D
# ============================================
print('\n--- Kaggle: GroundingDINO+SigLIP2 object features ---')
SIGLIP_HUB_PATH = kagglehub.dataset_download(
    'danielq07/causalvqa-groundingdino-siglip-feature'
)
print(f'GroundingDINO+SigLIP2 dataset root: {SIGLIP_HUB_PATH}')

def _flat_pkl_dir(root_path):
    counts = {}
    for pkl_path in Path(root_path).rglob('*.pkl'):
        counts[pkl_path.parent] = counts.get(pkl_path.parent, 0) + 1
    if not counts: return None, 0
    best_dir, count = max(counts.items(), key=lambda item: item[1])
    return str(best_dir), count

direct_pkl_dir, direct_pkl_count = _flat_pkl_dir(SIGLIP_HUB_PATH)
zip_files = sorted(Path(SIGLIP_HUB_PATH).rglob('*.zip'))
if direct_pkl_dir and direct_pkl_count >= 26000 and not zip_files:
    GDINO_MERGED_PATH = direct_pkl_dir
    print(f'Using unpacked SigLIP2 features directly: {direct_pkl_count} pkl')
else:
    GDINO_MERGED_PATH = ('/content/gdino_siglip2_merged' if os.path.exists('/content')
                         else '/kaggle/working/gdino_siglip2_merged')
    os.makedirs(GDINO_MERGED_PATH, exist_ok=True)
    for pkl_path in tqdm(list(Path(SIGLIP_HUB_PATH).rglob('*.pkl')), desc='Copy SigLIP2 pkls'):
        target = Path(GDINO_MERGED_PATH) / pkl_path.name
        if not target.exists(): shutil.copy2(pkl_path, target)
    # NB3 uploads two ZIP shards; Kaggle may preserve those ZIP files.
    for zip_path in zip_files:
        print(f'Extracting {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            members = [m for m in zf.namelist() if m.endswith('.pkl')]
            for member in tqdm(members, desc=f'Extract {zip_path.name[:32]}'):
                target = Path(GDINO_MERGED_PATH) / Path(member).name
                if target.exists(): continue
                with zf.open(member) as source, open(target, 'wb') as destination:
                    shutil.copyfileobj(source, destination)
siglip_pkl_count = len(list(Path(GDINO_MERGED_PATH).glob('*.pkl')))
if siglip_pkl_count == 0: raise FileNotFoundError(f'No SigLIP2 .pkl under {SIGLIP_HUB_PATH}')
print(f'GroundingDINO+SigLIP2 ready: {siglip_pkl_count} pkl in {GDINO_MERGED_PATH}')

# ============================================
# Merge DINOv3 features (train/val/test → flat dir)
# ============================================
print('\n=== Merge DINOv3 Features ===')
CLIP_MERGED_PATH = '/content/dinov3_T16_dim1024_merge' if os.path.exists('/content') else '/kaggle/working/dinov3_T16_dim1024_merge'
os.makedirs(CLIP_MERGED_PATH, exist_ok=True)

candidate_roots = [DINOV3_HUB_PATH, os.path.join(DINOV3_HUB_PATH, 'features')]

def has_split_subdirs(root):
    if not os.path.exists(root): return False
    names = set(os.listdir(root))
    return ('train' in names) and ('test' in names) and ('valid' in names or 'val' in names)

resolved_source = next((c for c in candidate_roots if has_split_subdirs(c)), None)

if resolved_source is None:
    maybe_merged = _find_best_dir(DINOV3_HUB_PATH, ['dinov3_T16_dim1024_merge', 'dinov3-feat'])
    pt_files = [f for f in os.listdir(maybe_merged) if f.endswith('.pt')] if (maybe_merged and os.path.exists(maybe_merged)) else []
    if pt_files:
        CLIP_MERGED_PATH = maybe_merged
        print(f'DINOv3 đã merged sẵn: {CLIP_MERGED_PATH} ({len(pt_files)} .pt)')
    else:
        raise FileNotFoundError(f'Không tìm thấy DINOv3 split folders: {candidate_roots}')
else:
    print(f'DINOv3 split source: {resolved_source}')
    total_copied = 0
    for split in ['train', 'test', 'valid', 'val']:
        split_folder = os.path.join(resolved_source, split)
        if not os.path.exists(split_folder): continue
        split_pt = [f for f in os.listdir(split_folder) if f.endswith('.pt')]
        print(f'{split}: {len(split_pt)} files')
        for fname in tqdm(split_pt, desc=f'Copying {split}'):
            src = os.path.join(split_folder, fname)
            dst = os.path.join(CLIP_MERGED_PATH, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst); total_copied += 1
    final_count = len([f for f in os.listdir(CLIP_MERGED_PATH) if f.endswith('.pt')])
    print(f'Merge complete: {final_count} .pt (copied {total_copied} new)')

# ============================================
# Resolved paths
# ============================================
ANNOTATION_QA_PATH = _find_best_dir(ANN_HUB_PATH, ['QA'])
SPLIT_TXT_PATH = _find_best_dir(SPLIT_HUB_PATH, ['split'])

print('\n📂 Resolved training paths:')
print(f'  CLIP_MERGED_PATH   : {CLIP_MERGED_PATH}')
print(f'  GDINO_MERGED_PATH  : {GDINO_MERGED_PATH}')
print(f'  ANNOTATION_QA_PATH : {ANNOTATION_QA_PATH}')
print(f'  SPLIT_TXT_PATH     : {SPLIT_TXT_PATH}')
print('Data download done.')

In [ ]:
# CELL 4: Core imports + NCOD/HUM + Verifier/Knowledge training functions
print('=== CELL 4: Imports + Functions (NCOD + HUM + Verifier/Knowledge) ===')

# Core training imports required by CELL 5 onward
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

from utils.util import set_seed, set_gpu_devices
import DataLoader as _dataset_module
# NB3 SigLIP2 schema; VideoQADataset methods resolve these globals at runtime.
_dataset_module.GDINO_ROI_DIM = 768
_dataset_module.GDINO_CLS_DIM = 768
_dataset_module.GDINO_BBOX_DIM = 4
_dataset_module.GDINO_OBJ_DIM = 1540
VideoQADataset = _dataset_module.VideoQADataset
from networks.model import VideoQAmodel

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

def train_epoch_integrated(
    model, optimizer_model, optimizer_u, U, loader, xe, bce, device, epoch, key_to_idx,
    accumulation_steps=1, hum_alpha=1.0, lambda_verifier=0.2, lambda_knowledge=0.1
):
    model.train()
    total_loss, total_l1, total_l2 = 0.0, 0.0, 0.0
    total_verifier, total_knowledge = 0.0, 0.0
    correct, total = 0, 0
    selector_count = 0
    selector_unique_sum = 0.0
    selector_overlap_sum = 0.0
    selector_span_sum = 0.0
    selector_time_sum = torch.zeros(model.frame_topK, device=device)
    optimizer_model.zero_grad()
    optimizer_u.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['logits']
        fused_logits = out.get('fused_score', logits)
        verifier_logits = out.get('verifier_logits', logits)
        knowledge_logits = out.get('knowledge_score', None)

        batch_selected_time = out['selected_time'].detach()
        batch_size_now = batch_selected_time.size(0)
        selector_count += batch_size_now
        selector_unique_sum += out['selection_unique_ratio'].detach().sum().item()
        selector_overlap_sum += out['selection_overlap'].detach().sum().item()
        selector_span_sum += out['selected_time_span'].detach().sum().item()
        selector_time_sum += batch_selected_time.sum(dim=0)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample

        hard_mask = (q_family_id >= 2)
        l1 = torch.where(hard_mask, hum_loss, lum_loss).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        if knowledge_logits is not None:
            knowledge_loss = bce(knowledge_logits, y_onehot)
        else:
            knowledge_loss = torch.tensor(0.0, device=device)

        model_loss = l1 + lambda_verifier * verifier_loss + lambda_knowledge * knowledge_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            optimizer_model.zero_grad()
            optimizer_u.step()
            optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item()
        total_l2 += l2.item()
        total_verifier += verifier_loss.item()
        total_knowledge += knowledge_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (fused_logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'l1': total_l1 / (batch_idx + 1),
            'l2': total_l2 / (batch_idx + 1),
            'ver': total_verifier / (batch_idx + 1),
            'know': total_knowledge / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100,
            'sel_unique': selector_unique_sum / max(selector_count, 1),
            'sel_overlap': selector_overlap_sum / max(selector_count, 1),
        })

    n = len(loader)
    selector_metrics = {
        'unique_ratio': selector_unique_sum / max(selector_count, 1),
        'overlap': selector_overlap_sum / max(selector_count, 1),
        'time_span': selector_span_sum / max(selector_count, 1),
        'selected_time_mean': (
            selector_time_sum / max(selector_count, 1)
        ).detach().cpu().tolist(),
    }
    return (
        total_loss / n,
        total_l1 / n,
        total_l2 / n,
        total_verifier / n,
        total_knowledge / n,
        correct / max(total, 1) * 100,
        selector_metrics,
    )

_QTYPE_SUFFIXES = [
    'counterfactual_reason', 'predictive_reason',
    'counterfactual', 'predictive', 'explanatory', 'descriptive',
]

def _split_qns_key(qns_key):
    key = str(qns_key)
    for qtype in _QTYPE_SUFFIXES:
        suffix = f'_{qtype}'
        if key.endswith(suffix):
            return key[:-len(suffix)], qtype
    return key, 'unknown'

def _compute_acc_all_metrics(type_results):
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason'),
    ]
    metrics = {}
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        metrics[name] = (
            sum(1 for row in rows if row['correct']) / len(rows) * 100.0
            if rows else 0.0
        )

    def paired_accuracy(answer_type, reason_type):
        answer_by_video = {
            row['video_id']: row['correct']
            for row in type_results.get(answer_type, [])
        }
        reason_by_video = {
            row['video_id']: row['correct']
            for row in type_results.get(reason_type, [])
        }
        common = set(answer_by_video) & set(reason_by_video)
        if not common:
            return 0.0
        return (
            sum(answer_by_video[vid] and reason_by_video[vid] for vid in common)
            / len(common) * 100.0
        )

    metrics['PAR'] = paired_accuracy('predictive', 'predictive_reason')
    metrics['CAR'] = paired_accuracy('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (
        metrics['Description'] + metrics['Explanation']
        + metrics['PAR'] + metrics['CAR']
    ) / 4.0
    return metrics

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    type_results = {}
    selector_count = 0
    selector_unique_sum = 0.0
    selector_overlap_sum = 0.0
    selector_span_sum = 0.0
    selector_time_sum = torch.zeros(model.frame_topK, device=device)
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            batch_selected_time = out['selected_time']
            batch_size_now = batch_selected_time.size(0)
            selector_count += batch_size_now
            selector_unique_sum += out['selection_unique_ratio'].sum().item()
            selector_overlap_sum += out['selection_overlap'].sum().item()
            selector_span_sum += out['selected_time_span'].sum().item()
            selector_time_sum += batch_selected_time.sum(dim=0)
            preds = logits.argmax(-1)
            correct += (preds == tgt).sum().item()
            total += tgt.size(0)

            for key, pred, target in zip(
                qns_keys, preds.detach().cpu().tolist(), tgt.detach().cpu().tolist()
            ):
                video_id, qtype = _split_qns_key(key)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'correct': bool(int(pred) == int(target)),
                })

    metrics = _compute_acc_all_metrics(type_results)
    metrics['Plain_Acc'] = correct / max(total, 1) * 100.0
    metrics['Selector_Unique_Ratio'] = selector_unique_sum / max(selector_count, 1)
    metrics['Selector_Overlap'] = selector_overlap_sum / max(selector_count, 1)
    metrics['Selector_Time_Span'] = selector_span_sum / max(selector_count, 1)
    metrics['Selector_Time_Mean'] = (
        selector_time_sum / max(selector_count, 1)
    ).cpu().tolist()
    return metrics

print('Imports and functions defined for integrated training.')

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports OK')

In [ ]:
# CELL 5: Setup Paths & Config
print('=== CELL 5: Paths & Config ===')

# ============================================
# DATA PATHS (resolved in CELL 3)
# ============================================
clip_merged_path = globals().get('CLIP_MERGED_PATH', None)
gdino_merged_path = globals().get('GDINO_MERGED_PATH', None)
annotation_qa_path = globals().get('ANNOTATION_QA_PATH', None)
split_txt_path = globals().get('SPLIT_TXT_PATH', None)

CLIP_FEATURE_PATH  = clip_merged_path  or '/kaggle/working/dinov3_T16_dim1024_merge'
GDINO_FEATURE_PATH = gdino_merged_path or '/content/gdino_siglip2_merged'
ANNOTATION_PATH    = annotation_qa_path or '/kaggle/input/text-annotation/QA'
SPLIT_DIR          = split_txt_path    or '/kaggle/input/casual-vid-data-split/split'

# Working directories
BASE = '/content/working' if os.path.exists('/content') else '/kaggle/working'
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'OK {name}: {items}')
        return True
    print(f'NOT FOUND {name}: {path}')
    return False

all_ok = True
all_ok &= verify_path('DINOv3 Features (1024D)', CLIP_FEATURE_PATH)
all_ok &= verify_path('GDINO+SigLIP2 Features (1540D)', GDINO_FEATURE_PATH)
all_ok &= verify_path('Annotations (QA)', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)

if not all_ok:
    raise FileNotFoundError('One or more required data paths are missing. Re-run CELL 3.')

# Sanity: count merged features
import glob as _glob
n_pt  = len(_glob.glob(os.path.join(CLIP_FEATURE_PATH, '*.pt')))
n_pkl = len(_glob.glob(os.path.join(GDINO_FEATURE_PATH, '*.pkl')))
print(f'\n📊 DINOv3 .pt: {n_pt} | GDINO+SigLIP2 .pkl: {n_pkl}')

# Validate a real feature package before constructing the large datasets.
import pickle as _pkl
_sample_pkl = next(iter(Path(GDINO_FEATURE_PATH).glob('*.pkl')), None)
if _sample_pkl is None: raise FileNotFoundError(f'No object feature .pkl in {GDINO_FEATURE_PATH}')
with open(_sample_pkl, 'rb') as _f: _sample_pkg = _pkl.load(_f)
_sample_frames = _sample_pkg.get('frames', [])
_sample_frame = next((fr for fr in _sample_frames if len(fr.get('roi_features', [])) > 0), None)
if _sample_frame is None: raise ValueError(f'No non-empty ROI frame in sample: {_sample_pkl.name}')
_roi_dim = np.asarray(_sample_frame['roi_features']).shape[-1]
_cls_dim = np.asarray(_sample_frame['class_text_embedding']).shape[-1]
if (_roi_dim, _cls_dim) != (768, 768):
    raise ValueError(f'Expected SigLIP2/DeBERTa (768,768), got ({_roi_dim},{_cls_dim}) in {_sample_pkl.name}')
print(f'✅ SigLIP2 schema: ROI={_roi_dim}, class={_cls_dim}, bbox=4, object={_roi_dim + _cls_dim + 4}D')
del _sample_pkg, _sample_frames, _sample_frame

# ============================================
# 3-RUN TUNING PRESETS
# ============================================
RUN_TRAINING = True
RUN_PROFILE = 'run1'
RUN3_REG_MODE = 'dropout'

RUN_PROFILES = {
    'baseline': {
        'epoch': 10, 'lr': 1e-5,
        'lambda_verifier': 0.3, 'lambda_knowledge': 0.2,
        'early_stop_start_epoch': 5, 'early_stop_patience': 4,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run1': {
        'epoch': 20, 'lr': 1e-5,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.3,
        'early_stop_start_epoch': 8, 'early_stop_patience': 6,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run2': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.3,
        'early_stop_start_epoch': 6, 'early_stop_patience': 5,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run3': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.3,
        'early_stop_start_epoch': 6, 'early_stop_patience': 5,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
}

if RUN_PROFILE not in RUN_PROFILES:
    raise ValueError(f'Invalid RUN_PROFILE={RUN_PROFILE}')

RUN_TAG = f'{RUN_PROFILE}_temporalrel_v1'
MODEL_FILENAME = f'best_model_gdinosiglip2_ncod_hum_{RUN_TAG}.ckpt'
LATEST_CKPT_FILENAME = f'latest_checkpoint_gdinosiglip2_ncod_hum_{RUN_TAG}.ckpt'
TRAIN_HISTORY_FILENAME = f'train_history_gdinosiglip2_ncod_hum_{RUN_TAG}.csv'
PREDICTIONS_CSV_FILENAME = f'test_predictions_gdinosiglip2_ncod_hum_{RUN_TAG}.csv'
METRICS_JSON_FILENAME = f'final_metrics_gdinosiglip2_ncod_hum_{RUN_TAG}.json'
BEST_ARTIFACT_NAME = f'best-model-gdinosiglip2-ncod-hum-{RUN_TAG}'
LAST_ARTIFACT_NAME = f'last-checkpoint-gdinosiglip2-ncod-hum-{RUN_TAG}'
FINAL_ARTIFACT_NAME = f'final-results-gdinosiglip2-ncod-hum-{RUN_TAG}'

FEAT_DIM = 1024  # DINOv3 frame
OBJ_DIM  = 1540  # SigLIP2 ROI(768) + DeBERTa cls(768) + bbox(4)
print(f'\nBackbone: DINOv3 ({FEAT_DIM}-d frame) + GroundingDINO+SigLIP2 ({OBJ_DIM}-d obj)')
print(f'Run profile: {RUN_PROFILE}')

class Config:
    # Paths
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    # Model architecture
    topK_frame = 16
    objs = 12              # giảm 20→12 match MAX_BOXES của GDINO notebook
    frames = 16
    select_frames = 5
    topK_obj = 12          # unused vì use_grounding_dino=True (model skip obj_sorter)

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = OBJ_DIM     # 1540
    use_grounding_dino = True

    # Top-5 Sparse Temporal Relation v1
    use_temporal_relation = True
    temporal_relation_layers = 1
    temporal_relation_nheads = 8
    temporal_relation_ffn = 1024
    temporal_relation_dropout = 0.2
    coverage_loss_enabled = False
    lambda_coverage = 0.01  # reserved for a later ablation

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    # Text encoder (DeBERTa v1, no LoRA, full fine-tuning)
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 2e-6
    text_pool_mode = 1
    use_lora = False

    # Training
    bs = 32
    accumulation_steps = 1
    lr = 1e-5
    epoch = 20
    gpu = 0
    gamma = 0.5
    decay = 1e-4
    n_query = 5
    lambda_verifier = 0.3
    lambda_knowledge = 0.2
    return_family_id = True

    # LR scheduler + early stopping
    lr_patience = 2
    min_lr = 1e-7
    early_stop_patience = 4
    early_stop_min_delta = 0.05
    early_stop_start_epoch = 5

    # NCOD + HUM
    ncod_u_lr = 0.1
    hum_alpha = 1.0

    # Other
    hard_eval = True
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    # Spawn + persistent workers avoids Colab/Python 3.12 parent-PID cleanup errors.
    num_workers = 2

# Apply selected preset overrides
for _k, _v in RUN_PROFILES[RUN_PROFILE].items():
    setattr(Config, _k, _v)

if RUN_PROFILE == 'run3':
    if RUN3_REG_MODE == 'dropout':
        Config.dropout = 0.25
        Config.encoder_dropout = 0.25
    elif RUN3_REG_MODE == 'decay':
        Config.decay = 8e-5
    else:
        raise ValueError(f'Invalid RUN3_REG_MODE={RUN3_REG_MODE}')

args = Config()

if args.bs != 32 or args.accumulation_steps != 1:
    raise ValueError('This notebook is locked to physical/effective batch size 32 (bs=32, accumulation=1).')

if args.text_encoder_type != 'microsoft/deberta-base':
    raise ValueError('Train notebook uses DeBERTa v1 only.')

set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Run tag: {RUN_TAG}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}, objs={args.objs}')
print(f'use_grounding_dino={args.use_grounding_dino} → obj_sorter SKIPPED, LayerNorm enabled trên obj features')
print(f'Effective bs: physical={args.bs} × accum={args.accumulation_steps} = {args.bs * args.accumulation_steps}')
print(f'lr={args.lr} | decay={args.decay} | verifier={args.lambda_verifier} | knowledge={args.lambda_knowledge}')
print(f'Early stop: start={args.early_stop_start_epoch}, patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta}')
print(f'Model file: {MODEL_FILENAME}')

In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)

loader_kwargs = {
    'batch_size': args.bs,
    'num_workers': args.num_workers,
    'pin_memory': torch.cuda.is_available(),
}
# persistent_workers is invalid when num_workers=0 and unnecessary on Colab.
if args.num_workers > 0:
    loader_kwargs.update({
        'persistent_workers': True,
        'prefetch_factor': 2,
        'multiprocessing_context': 'spawn',
    })

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

train_sample_keys = [f"{row.video_id}_{row.type}" for row in train_ds.sample_list.itertuples(index=False)]
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# CELL 7: Model + Optimizers + NCOD U (No LoRA, full DeBERTa v1 tuning)
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
model.to(device)
if not hasattr(model, 'temporal_relation'):
    raise RuntimeError(
        'Updated networks/model.py with SparseTemporalRelation is required. '
        'Push/pull the temporalrel_v1 code before running this notebook.'
    )
temporal_param_count = sum(p.numel() for p in model.temporal_relation.parameters())
print(f'Temporal relation params: {temporal_param_count / 1e6:.3f}M')

non_text_params = []
text_base_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'text_encoder' in name:
        text_base_params.append(p)
    else:
        non_text_params.append(p)

param_groups = []
if len(non_text_params) > 0:
    param_groups.append({'params': non_text_params, 'lr': args.lr, 'weight_decay': args.decay})
if len(text_base_params) > 0:
    param_groups.append({'params': text_base_params, 'lr': args.text_encoder_lr, 'weight_decay': args.decay})
if len(param_groups) == 0:
    raise RuntimeError('No trainable parameters found for optimizer_model.')

optimizer_model = torch.optim.AdamW(param_groups)
scheduler = ReduceLROnPlateau(
    optimizer_model,
    mode='max',
    factor=args.gamma,
    patience=args.lr_patience,
    threshold=args.early_stop_min_delta,
    threshold_mode='abs',
    min_lr=args.min_lr
)

U = torch.nn.Parameter(torch.full((len(train_ds),), 1e-8, dtype=torch.float32, device=device))
optimizer_u = torch.optim.SGD([U], lr=args.ncod_u_lr)

xe = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')
print(f'Text-encoder trainable params: {sum(p.numel() for p in text_base_params)/1e6:.3f}M')
print(f'U shape: {tuple(U.shape)}')
print(f'Artifacts: best={BEST_ARTIFACT_NAME} | latest={LAST_ARTIFACT_NAME}')

In [ ]:
# CELL 8: Init W&B + Resume Checkpoint
print('=== CELL 8: Initialize W&B Run ===')

start_epoch = 1
best_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
history_rows = []

LATEST_CKPT_PATH = os.path.join(MODEL_DIR, LATEST_CKPT_FILENAME)
TRAIN_HISTORY_CSV_PATH = os.path.join(MODEL_DIR, TRAIN_HISTORY_FILENAME)

# For clean 3-run tuning, default to fresh training.
# Set True only if you intentionally continue the same RUN_TAG.
RESUME_FROM_CHECKPOINT = False
RESUME_SOURCE = 'wandb'
RESUME_ARTIFACT_ALIAS = 'latest'

wandb_config = {
    'run_tag': RUN_TAG,
    'run_profile': RUN_PROFILE,
    'backbone': 'dinov3+groundingdino-siglip2',
    'text_encoder': args.text_encoder_type,
    'use_lora': args.use_lora,
    'full_text_finetune': not args.freeze_text_encoder,
    'physical_batch_size': args.bs,
    'accumulation_steps': args.accumulation_steps,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'epochs': args.epoch,
    'use_temporal_relation': args.use_temporal_relation,
    'temporal_relation_layers': args.temporal_relation_layers,
    'temporal_relation_nheads': args.temporal_relation_nheads,
    'temporal_relation_ffn': args.temporal_relation_ffn,
    'temporal_relation_dropout': args.temporal_relation_dropout,
    'coverage_loss_enabled': args.coverage_loss_enabled,
    'lambda_coverage_reserved': args.lambda_coverage,
    'lambda_verifier': args.lambda_verifier,
    'lambda_knowledge': args.lambda_knowledge,
    'ncod_u_lr': args.ncod_u_lr,
    'hum_alpha': args.hum_alpha,
    'lr_main': args.lr,
    'lr_text_encoder': args.text_encoder_lr,
    'lr_scheduler_factor': args.gamma,
    'lr_scheduler_patience': args.lr_patience,
    'min_lr': args.min_lr,
    'early_stop_patience': args.early_stop_patience,
    'early_stop_min_delta': args.early_stop_min_delta,
    'early_stop_start_epoch': args.early_stop_start_epoch,
    'resume_enabled': RESUME_FROM_CHECKPOINT,
    'resume_source': RESUME_SOURCE,
    'checkpoint_upload_policy': 'finish_only_last_and_best'
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, config=wandb_config, reinit=True)
wandb.watch(model, log='gradients', log_freq=100)
print(f'W&B run: {run.url}')

def _load_resume_checkpoint(path, map_location):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    return torch.load(path, map_location=map_location)

if RESUME_FROM_CHECKPOINT:
    print(f'Resume enabled from: {RESUME_SOURCE}')
    try:
        checkpoint = None
        resume_path = None

        api = wandb.Api()
        resume_entity = WANDB_ENTITY or api.default_entity
        artifact_path = f'{resume_entity}/{WANDB_PROJECT}/{LAST_ARTIFACT_NAME}:{RESUME_ARTIFACT_ALIAS}'
        print(f'Downloading W&B artifact: {artifact_path}')
        artifact = api.artifact(artifact_path)
        artifact_dir = artifact.download()
        candidate_path = os.path.join(artifact_dir, LATEST_CKPT_FILENAME)
        if os.path.exists(candidate_path):
            resume_path = candidate_path
        else:
            ckpt_files = [f for f in os.listdir(artifact_dir) if f.endswith('.ckpt')]
            if not ckpt_files:
                raise FileNotFoundError(f'No .ckpt found in W&B artifact: {artifact_dir}')
            resume_path = os.path.join(artifact_dir, ckpt_files[0])
        checkpoint = _load_resume_checkpoint(resume_path, device)

        ckpt_state = checkpoint['model_state_dict']
        model_state = model.state_dict()
        filtered_state = {}
        skipped_keys = []
        for k, v in ckpt_state.items():
            if k in model_state and v.shape != model_state[k].shape:
                skipped_keys.append(f'{k}: ckpt={list(v.shape)} vs model={list(model_state[k].shape)}')
            else:
                filtered_state[k] = v

        if skipped_keys:
            print(f'⚠️ Skipped {len(skipped_keys)} keys due to shape mismatch:')
            for sk in skipped_keys:
                print(f'  {sk}')
            print('These layers will be randomly initialized. Training from epoch 1.')

        missing, unexpected = model.load_state_dict(filtered_state, strict=False)
        if missing:
            print(f'Warning: missing model keys when resume: {len(missing)}')
        if unexpected:
            print(f'Warning: unexpected model keys when resume: {len(unexpected)}')

        if not skipped_keys:
            if 'optimizer_model_state_dict' in checkpoint:
                optimizer_model.load_state_dict(checkpoint['optimizer_model_state_dict'])
            if 'optimizer_u_state_dict' in checkpoint:
                optimizer_u.load_state_dict(checkpoint['optimizer_u_state_dict'])
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

            if 'U' in checkpoint:
                with torch.no_grad():
                    u_ckpt = checkpoint['U'].to(device).float().view(-1)
                    n = min(u_ckpt.numel(), U.numel())
                    U[:n].copy_(u_ckpt[:n])
                    if u_ckpt.numel() != U.numel():
                        print(f'Warning: U size mismatch (ckpt={u_ckpt.numel()}, current={U.numel()}); copied first {n}')

            best_acc = float(checkpoint.get('best_acc', 0.0))
            start_epoch = int(checkpoint.get('epoch', 0)) + 1
            best_epoch = int(checkpoint.get('best_epoch', 0))
            epochs_without_improvement = int(checkpoint.get('epochs_without_improvement', 0))
            history = checkpoint.get('history', history)
            history_rows = checkpoint.get('history_rows', history_rows)

            if os.path.exists(TRAIN_HISTORY_CSV_PATH):
                try:
                    history_rows = pd.read_csv(TRAIN_HISTORY_CSV_PATH).to_dict('records')
                    print(f'Loaded history CSV with {len(history_rows)} rows')
                except Exception as csv_err:
                    print(f'Warning: failed to load history CSV: {csv_err}')
        else:
            print('Optimizer/scheduler/U NOT restored due to architecture change. Starting fresh training state.')

        print(f'Resumed from: {resume_path}')
        print(f'Start epoch: {start_epoch} | Best acc: {best_acc:.2f}% | Best epoch: {best_epoch}')
        wandb.run.summary['resume_path'] = str(resume_path)
        wandb.run.summary['resume_start_epoch'] = int(start_epoch)
        wandb.run.summary['resume_best_acc'] = float(best_acc)
        wandb.run.summary['resume_best_epoch'] = int(best_epoch)
    except Exception as e:
        print(f'Warning: resume failed, starting from scratch. Error: {e}')
else:
    print('Resume disabled. Training starts from epoch 1.')

In [ ]:
# CELL 9: Integrated Training Loop + Checkpoint/CSV Logging + Early Stopping
print('=== CELL 9: Training ===')

if RUN_TRAINING:
    stop_training = False
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')
        (
            total_loss, l1, l2, verifier_loss, knowledge_loss,
            train_acc, selector_metrics,
        ) = train_epoch_integrated(
            model=model,
            optimizer_model=optimizer_model,
            optimizer_u=optimizer_u,
            U=U,
            loader=train_loader,
            xe=xe,
            bce=bce,
            device=device,
            epoch=ep,
            key_to_idx=train_key_to_idx,
            accumulation_steps=args.accumulation_steps,
            hum_alpha=args.hum_alpha,
            lambda_verifier=args.lambda_verifier,
            lambda_knowledge=args.lambda_knowledge
        )

        val_metrics = eval_epoch(model, val_loader, device)
        val_acc = float(val_metrics['Acc_ALL'])
        val_plain_acc = float(val_metrics['Plain_Acc'])
        scheduler.step(val_acc)

        history['train_loss'].append(total_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        current_lrs = [pg['lr'] for pg in optimizer_model.param_groups]
        min_lr_now = float(min(current_lrs))
        max_lr_now = float(max(current_lrs))

        improved = val_acc > (best_acc + args.early_stop_min_delta)
        if improved:
            best_acc = val_acc
            best_epoch = ep
            epochs_without_improvement = 0
            print(
                f'New best val Acc_ALL={best_acc:.2f}% at epoch {best_epoch} | '
                f'Plain={val_plain_acc:.2f}% | PAR={val_metrics["PAR"]:.2f}% | '
                f'CAR={val_metrics["CAR"]:.2f}%'
            )
        elif ep >= args.early_stop_start_epoch:
            epochs_without_improvement += 1
            print(
                f'No significant improvement for {epochs_without_improvement} epoch(s) '
                f'(patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta})'
            )

        epoch_row = {
            'epoch': ep,
            'train_total_loss': float(total_loss),
            'train_l1': float(l1),
            'train_l2': float(l2),
            'train_verifier_loss': float(verifier_loss),
            'train_knowledge_loss': float(knowledge_loss),
            'train_acc': float(train_acc),
            'val_acc': float(val_acc),
            'val/Acc_ALL': float(val_metrics['Acc_ALL']),
            'val/Plain_Acc': float(val_metrics['Plain_Acc']),
            'val/Description': float(val_metrics['Description']),
            'val/Explanation': float(val_metrics['Explanation']),
            'val/Predictive_Answer': float(val_metrics['Predictive-Answer']),
            'val/Predictive_Reason': float(val_metrics['Predictive-Reason']),
            'val/Counterfactual_Answer': float(val_metrics['Counterfactual-Answer']),
            'val/Counterfactual_Reason': float(val_metrics['Counterfactual-Reason']),
            'val/PAR': float(val_metrics['PAR']),
            'val/CAR': float(val_metrics['CAR']),
            'selector/train_unique_ratio': float(selector_metrics['unique_ratio']),
            'selector/train_overlap': float(selector_metrics['overlap']),
            'selector/train_time_span': float(selector_metrics['time_span']),
            'selector/val_unique_ratio': float(val_metrics['Selector_Unique_Ratio']),
            'selector/val_overlap': float(val_metrics['Selector_Overlap']),
            'selector/val_time_span': float(val_metrics['Selector_Time_Span']),
            'u_mean': float(U.detach().mean().item()),
            'u_max': float(U.detach().max().item()),
            'lr_main_min': min_lr_now,
            'lr_main_max': max_lr_now,
            'best_acc_so_far': float(best_acc),
            'best_epoch_so_far': int(best_epoch),
            'epochs_without_improvement': int(epochs_without_improvement)
        }
        for slot_idx, slot_time in enumerate(selector_metrics['selected_time_mean']):
            epoch_row[f'selector/train_time_slot_{slot_idx}'] = float(slot_time)
        for slot_idx, slot_time in enumerate(val_metrics['Selector_Time_Mean']):
            epoch_row[f'selector/val_time_slot_{slot_idx}'] = float(slot_time)

        history_rows.append(epoch_row)
        pd.DataFrame(history_rows).to_csv(TRAIN_HISTORY_CSV_PATH, index=False)

        wandb.log(epoch_row)

        ckpt = {
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_model_state_dict': optimizer_model.state_dict(),
            'optimizer_u_state_dict': optimizer_u.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': best_acc,
            'best_epoch': best_epoch,
            'epochs_without_improvement': epochs_without_improvement,
            'history': history,
            'history_rows': history_rows,
            'U': U.detach().cpu(),
            'train_sample_keys': train_sample_keys
        }

        torch.save(ckpt, LATEST_CKPT_PATH)

        # Keep overwriting local files during training. Upload only once in CELL 11.
        if improved:
            torch.save(ckpt, save_path)

        if ep >= args.early_stop_start_epoch and epochs_without_improvement >= args.early_stop_patience:
            print(f'Early stopping at epoch {ep}. Best val Acc_ALL={best_acc:.2f}% at epoch {best_epoch}.')
            wandb.run.summary['early_stopped'] = True
            wandb.run.summary['early_stop_epoch'] = int(ep)
            stop_training = True
            break

    wandb.run.summary['best_val_acc'] = float(best_acc)
    wandb.run.summary['best_epoch'] = int(best_epoch)

    if os.path.exists(save_path):
        best_ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(best_ckpt['model_state_dict'], strict=False)
        print(f'Loaded best checkpoint from epoch {best_epoch} for final evaluation.')

    if not stop_training:
        print(f'Training finished all {args.epoch} epochs. Best Val Acc_ALL: {best_acc:.2f}%')
else:
    print('Skipping training')

In [ ]:
# CELL 10: Detailed Evaluation + Memory Post-check + CSV export
print('=== CELL 10: Detailed Evaluation + Memory Post-check ===')
import seaborn as sns
from networks.knowledge_retriever import CausalKnowledgeRetriever

CSV_OUTPUT_PATH = os.path.join(MODEL_DIR, PREDICTIONS_CSV_FILENAME)
COMPARISON_CSV_PATH = os.path.join(MODEL_DIR, 'run_comparison_gdinosiglip2_3run.csv')

# ===== Simple memory post-check config =====
TOPK_KNOWLEDGE = 5
MEMORY_PASS_THRESHOLD = 0.15
MEMORY_GATE_ENABLED = True
MEMORY_MARGIN = 0.05

def _resolve_kb_path():
    candidates = [
        os.path.join(os.getcwd(), 'data', 'causal_knowledge_bank.json'),
        '/content/tranSTR_Casual/causalvid/data/causal_knowledge_bank.json',
        '/kaggle/working/tranSTR_Casual/causalvid/data/causal_knowledge_bank.json',
        '/kaggle/working/causalvid/data/causal_knowledge_bank.json'
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

KB_PATH = _resolve_kb_path()
retriever = CausalKnowledgeRetriever(KB_PATH, topk=TOPK_KNOWLEDGE) if KB_PATH else None
print(f'Knowledge bank path: {KB_PATH if KB_PATH else "NOT FOUND (memory check disabled)"}')

def _qtype_to_family(qtype):
    qtype = str(qtype)
    valid = {'descriptive', 'explanatory', 'predictive', 'predictive_reason', 'counterfactual', 'counterfactual_reason'}
    return qtype if qtype in valid else 'descriptive'

def _score_candidate_with_memory(question, candidate, q_family, video_anchors=None):
    if retriever is None:
        return 0.0, []
    hits = retriever.retrieve(
        question=str(question),
        candidate=str(candidate),
        video_anchors=video_anchors or [],
        q_family=str(q_family),
        topk=TOPK_KNOWLEDGE
    )
    top_score = max([float(h.get('score', 0.0)) for h in hits], default=0.0)
    return top_score, hits

def _build_eval_meta_map(loader):
    dataset = getattr(loader, 'dataset', None)
    sample_list = getattr(dataset, 'sample_list', None) if dataset is not None else None
    meta_map = {}
    if sample_list is None:
        return meta_map

    for _, row in sample_list.iterrows():
        vid = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        qns_key = f'{vid}_{qtype}'
        meta_map[qns_key] = {
            'video_id': vid,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)]
        }
    return meta_map

def evaluate_detailed_v2(model, loader, device, log_to_wandb=True):
    model.eval()
    type_results = {}
    prediction_rows = []
    meta_map = _build_eval_meta_map(loader)

    memory_match_flags = []
    memory_pass_flags = []
    memory_gate_correct_flags = []

    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None

            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=-1)
            targets = ans_id.numpy()

            for i, (key, pred, target, prob_vec) in enumerate(zip(qns_keys, preds, targets, probs)):
                meta = meta_map.get(str(key), {})
                qtype = str(meta.get('question_type', 'unknown'))
                q_family = _qtype_to_family(qtype)
                video_id = str(meta.get('video_id', str(key)))
                question = str(meta.get('question', qns[i]))
                answers = meta.get('answers', ['', '', '', '', ''])
                if len(answers) < 5:
                    answers += [''] * (5 - len(answers))
                answers = answers[:5]

                correct_idx = int(target)
                predicted_idx = int(pred)
                is_correct = int(correct_idx == predicted_idx)

                # Post-check prediction against memory support
                cand_mem_scores = []
                for cand in answers:
                    score, _ = _score_candidate_with_memory(question, cand, q_family, video_anchors=[video_id])
                    cand_mem_scores.append(float(score))

                mem_best_idx = int(np.argmax(cand_mem_scores)) if len(cand_mem_scores) > 0 else predicted_idx
                pred_mem_score = float(cand_mem_scores[predicted_idx]) if len(cand_mem_scores) > predicted_idx else 0.0
                gt_mem_score = float(cand_mem_scores[correct_idx]) if len(cand_mem_scores) > correct_idx else 0.0

                memory_match_pred = int(predicted_idx == mem_best_idx)
                memory_pass_pred = int(pred_mem_score >= MEMORY_PASS_THRESHOLD)

                gated_idx = predicted_idx
                if MEMORY_GATE_ENABLED and len(cand_mem_scores) > 0:
                    if (cand_mem_scores[mem_best_idx] - pred_mem_score) >= MEMORY_MARGIN:
                        gated_idx = mem_best_idx
                gated_correct = int(gated_idx == correct_idx)

                memory_match_flags.append(memory_match_pred)
                memory_pass_flags.append(memory_pass_pred)
                memory_gate_correct_flags.append(gated_correct)

                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'pred': predicted_idx,
                    'target': correct_idx,
                    'correct': bool(is_correct)
                })

                prediction_rows.append({
                    'video_id': video_id,
                    'question_type': qtype,
                    'question': question,
                    'correct_idx': correct_idx,
                    'predicted_idx': predicted_idx,
                    'predicted_idx_after_memory_gate': int(gated_idx),
                    'is_correct': is_correct,
                    'is_correct_after_memory_gate': gated_correct,
                    'confidence': float(prob_vec[predicted_idx]),
                    'memory_best_idx': mem_best_idx,
                    'memory_match_pred': memory_match_pred,
                    'memory_pass_pred': memory_pass_pred,
                    'memory_pred_score': pred_mem_score,
                    'memory_gt_score': gt_mem_score,
                    'a0': answers[0],
                    'prob_a0': float(prob_vec[0]),
                    'a1': answers[1],
                    'prob_a1': float(prob_vec[1]),
                    'a2': answers[2],
                    'prob_a2': float(prob_vec[2]),
                    'a3': answers[3],
                    'prob_a3': float(prob_vec[3]),
                    'a4': answers[4],
                    'prob_a4': float(prob_vec[4]),
                    'predicted_answer': answers[predicted_idx],
                    'correct_answer': answers[correct_idx]
                })

    prediction_df = pd.DataFrame(prediction_rows)
    prediction_df.to_csv(CSV_OUTPUT_PATH, index=False)
    print(f'Saved detailed predictions CSV: {CSV_OUTPUT_PATH}')

    metrics = {}
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason')
    ]
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        total = len(rows)
        correct = sum(1 for r in rows if r['correct'])
        metrics[name] = (correct / total * 100) if total > 0 else 0.0

    def _calc_hard_metric(type_ans, type_reason):
        if type_ans not in type_results or type_reason not in type_results:
            return 0.0
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results[type_ans]}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results[type_reason]}
        common_vids = set(ans_by_vid.keys()) & set(reason_by_vid.keys())
        if len(common_vids) == 0:
            return 0.0
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        return both_correct / len(common_vids) * 100

    metrics['PAR'] = _calc_hard_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _calc_hard_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (
        metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']
    ) / 4.0

    # New memory post-check metrics
    metrics['Memory_Consistency'] = float(np.mean(memory_match_flags) * 100) if len(memory_match_flags) > 0 else 0.0
    metrics['Memory_Pass_Rate'] = float(np.mean(memory_pass_flags) * 100) if len(memory_pass_flags) > 0 else 0.0
    metrics['Memory_Gated_Acc'] = float(np.mean(memory_gate_correct_flags) * 100) if len(memory_gate_correct_flags) > 0 else 0.0

    # Weighted score for 3-run selection (weak-group priority)
    metrics['WeightedScore_WeakPriority'] = (
        0.35 * metrics['Predictive-Reason'] +
        0.35 * metrics['Counterfactual-Reason'] +
        0.20 * metrics['Acc_ALL'] +
        0.10 * float(best_acc)
    )

    if log_to_wandb:
        wandb.log({
            'eval/Description': metrics['Description'],
            'eval/Explanation': metrics['Explanation'],
            'eval/Predictive_Answer': metrics['Predictive-Answer'],
            'eval/Predictive_Reason': metrics['Predictive-Reason'],
            'eval/Counterfactual_Answer': metrics['Counterfactual-Answer'],
            'eval/Counterfactual_Reason': metrics['Counterfactual-Reason'],
            'eval/PAR': metrics['PAR'],
            'eval/CAR': metrics['CAR'],
            'eval/Acc_ALL': metrics['Acc_ALL'],
            'eval/Memory_Consistency': metrics['Memory_Consistency'],
            'eval/Memory_Pass_Rate': metrics['Memory_Pass_Rate'],
            'eval/Memory_Gated_Acc': metrics['Memory_Gated_Acc'],
            'eval/WeightedScore_WeakPriority': metrics['WeightedScore_WeakPriority']
        })

    print(f"PAR: {metrics['PAR']:.2f}% | CAR: {metrics['CAR']:.2f}% | Acc_ALL: {metrics['Acc_ALL']:.2f}%")
    print(f"Memory_Consistency: {metrics['Memory_Consistency']:.2f}% | Memory_Gated_Acc: {metrics['Memory_Gated_Acc']:.2f}%")
    print(f"WeightedScore_WeakPriority: {metrics['WeightedScore_WeakPriority']:.2f}")
    return metrics, type_results, CSV_OUTPUT_PATH

metrics, raw_results, predictions_csv = evaluate_detailed_v2(model, test_loader, device, log_to_wandb=True)

# Append/update run comparison table
comparison_row = {
    'run_tag': RUN_TAG,
    'run_profile': RUN_PROFILE,
    'best_val_acc': float(best_acc),
    'best_epoch': int(best_epoch),
    'Description': float(metrics.get('Description', 0.0)),
    'Explanation': float(metrics.get('Explanation', 0.0)),
    'Predictive-Answer': float(metrics.get('Predictive-Answer', 0.0)),
    'Predictive-Reason': float(metrics.get('Predictive-Reason', 0.0)),
    'Counterfactual-Answer': float(metrics.get('Counterfactual-Answer', 0.0)),
    'Counterfactual-Reason': float(metrics.get('Counterfactual-Reason', 0.0)),
    'PAR': float(metrics.get('PAR', 0.0)),
    'CAR': float(metrics.get('CAR', 0.0)),
    'Acc_ALL': float(metrics.get('Acc_ALL', 0.0)),
    'WeightedScore_WeakPriority': float(metrics.get('WeightedScore_WeakPriority', 0.0)),
}

if os.path.exists(COMPARISON_CSV_PATH):
    comp_df = pd.read_csv(COMPARISON_CSV_PATH)
    comp_df = comp_df[comp_df['run_tag'] != RUN_TAG]
    comp_df = pd.concat([comp_df, pd.DataFrame([comparison_row])], ignore_index=True)
else:
    comp_df = pd.DataFrame([comparison_row])

comp_df = comp_df.sort_values(by='WeightedScore_WeakPriority', ascending=False)
comp_df.to_csv(COMPARISON_CSV_PATH, index=False)
print(f'Saved/updated run comparison CSV: {COMPARISON_CSV_PATH}')
print(comp_df[['run_tag', 'run_profile', 'best_val_acc', 'Predictive-Reason', 'Counterfactual-Reason', 'Acc_ALL', 'WeightedScore_WeakPriority']])

wandb.run.summary['run_tag'] = RUN_TAG
wandb.run.summary['run_profile'] = RUN_PROFILE
wandb.run.summary['weighted_score_weak_priority'] = float(metrics['WeightedScore_WeakPriority'])
print(metrics)

In [ ]:
# CELL 11: Finish W&B
print('=== CELL 11: Finish W&B ===')

METRICS_JSON_PATH = os.path.join(MODEL_DIR, METRICS_JSON_FILENAME)
with open(METRICS_JSON_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved metrics JSON: {METRICS_JSON_PATH}')

UPLOAD_CKPT_ARTIFACTS_AT_FINISH = True
if UPLOAD_CKPT_ARTIFACTS_AT_FINISH:
    if os.path.exists(LATEST_CKPT_PATH):
        latest_ckpt_artifact = wandb.Artifact(
            name=LAST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'latest',
                'run_tag': RUN_TAG,
                'run_profile': RUN_PROFILE,
                'text_encoder': args.text_encoder_type,
                'lora': args.use_lora,
                'ncod_hum': True
            }
        )
        latest_ckpt_artifact.add_file(LATEST_CKPT_PATH, name=LATEST_CKPT_FILENAME)
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            latest_ckpt_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)
        wandb.log_artifact(latest_ckpt_artifact, aliases=['latest', 'finish'])
        print('Uploaded latest checkpoint artifact to W&B.')
    else:
        print(f'Warning: latest checkpoint not found at {LATEST_CKPT_PATH}')

    if os.path.exists(save_path):
        best_ckpt_artifact = wandb.Artifact(
            name=BEST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'best',
                'run_tag': RUN_TAG,
                'run_profile': RUN_PROFILE,
                'text_encoder': args.text_encoder_type,
                'lora': args.use_lora,
                'ncod_hum': True
            }
        )
        best_ckpt_artifact.add_file(save_path, name=MODEL_FILENAME)
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            best_ckpt_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)
        wandb.log_artifact(best_ckpt_artifact, aliases=['best', 'finish'])
        print('Uploaded best checkpoint artifact to W&B.')
    else:
        print(f'Warning: best checkpoint not found at {save_path}')

final_artifact = wandb.Artifact(
    name=FINAL_ARTIFACT_NAME,
    type='results',
    metadata={
        'run_tag': RUN_TAG,
        'run_profile': RUN_PROFILE,
        'backbone': 'dinov3+groundingdino-siglip2',
        'text_encoder': args.text_encoder_type,
        'lora': args.use_lora,
        'ncod_hum': True
    }
)
if os.path.exists(METRICS_JSON_PATH):
    final_artifact.add_file(METRICS_JSON_PATH, name=METRICS_JSON_FILENAME)
if os.path.exists(predictions_csv):
    final_artifact.add_file(predictions_csv, name=PREDICTIONS_CSV_FILENAME)
if os.path.exists(TRAIN_HISTORY_CSV_PATH):
    final_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)

comparison_csv_path = os.path.join(MODEL_DIR, 'run_comparison_gdinosiglip2_3run.csv')
if os.path.exists(comparison_csv_path):
    final_artifact.add_file(comparison_csv_path, name='run_comparison_gdinosiglip2_3run.csv')

wandb.log_artifact(final_artifact)
wandb.finish()
print('W&B finished: uploaded exactly one last checkpoint, one best checkpoint, and lightweight result files.')

In [ ]:
# CELL 12: Auto disconnect Colab runtime after training
# This cell runs only after W&B uploads and wandb.finish() in the previous cell.
import os
import time

print('✅ Training and W&B artifact uploads completed.')
print('Disconnecting Colab runtime in 15 seconds...')
time.sleep(15)

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as exc:
    print(f'google.colab.runtime.unassign() unavailable ({exc}); stopping process instead.')
    os.kill(os.getpid(), 9)
